# Preprocessing delle fonti (Op. 5, 7, 9–11, 14)

Pulizia delle quattro fonti e collegamento dei giocatori agli identificativi del dataset 1. I risultati vanno in `data/processed/` e servono al notebook 02.

**Dati necessari** in `data/raw/` (non inclusi nella repository, vedi README): `dataset_1/worldcup.db`, `dataset_2/`, `dataset_3/`, `dataset_4/Images/Images/`.

**Pacchetti:** `pandas`, `pyarrow`, `rapidfuzz`, `unidecode`. Le celle vanno eseguite in ordine.

## Dataset 1
Database SQLite aperto in sola lettura; elenco delle chiavi esterne di ogni tabella, usato per la bozza dell'ontologia.

In [ ]:
# [Cella 1] Connessione al database del dataset 1 (sola lettura) e chiavi esterne delle tabelle
# Librerie: sqlite3 legge il database del dataset 1, pandas gestisce le tabelle
import sqlite3, pandas as pd
from pathlib import Path

# Radice del progetto: se il notebook gira in notebooks/, si sale di un livello
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB = ROOT / "data" / "raw" / "dataset_1" / "worldcup.db"

# Ci si ferma subito se il file non c'è (errore tipico: percorso relativo sbagliato)
assert DB.exists(), f"Non trovo {DB}. Contenuto della cartella: {list(DB.parent.glob('*'))}"

con = sqlite3.connect(DB.as_uri() + "?mode=ro", uri=True)   # sola lettura
# Chiavi esterne di ogni tabella: dicono quali tabelle sono entità e quali relazioni
for t in pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con).name:
    fk = pd.read_sql(f"PRAGMA foreign_key_list('{t}')", con)
    print(t, fk[["from", "table", "to"]].values.tolist())

tournaments []
confederations []
teams [['confederation_id', 'confederations', 'confederation_id']]
players []
managers []
referees [['confederation_id', 'confederations', 'confederation_id']]
stadiums []
matches [['away_team_id', 'teams', 'team_id'], ['home_team_id', 'teams', 'team_id'], ['stadium_id', 'stadiums', 'stadium_id'], ['tournament_id', 'tournaments', 'tournament_id']]
awards []
qualified_teams [['team_id', 'teams', 'team_id'], ['tournament_id', 'tournaments', 'tournament_id']]
squads [['player_id', 'players', 'player_id'], ['team_id', 'teams', 'team_id'], ['tournament_id', 'tournaments', 'tournament_id']]
manager_appointments [['manager_id', 'managers', 'manager_id'], ['team_id', 'teams', 'team_id'], ['tournament_id', 'tournaments', 'tournament_id']]
referee_appointments [['referee_id', 'referees', 'referee_id'], ['tournament_id', 'tournaments', 'tournament_id']]
team_appearances [['opponent_id', 'teams', 'team_id'], ['team_id', 'teams', 'team_id'], ['match_id', 'matches', 

## Dataset 2
22 file annuali uniti in una tabella; nomi delle nazioni allineati al dataset 1 con 5 alias; sistema dei punti dell'epoca.

In [ ]:
# [Cella 2] Dataset 2: unione dei 22 file annuali, alias delle nazioni, sistema dei punti
import re, pandas as pd
from pathlib import Path

# Cartella dei 22 file annuali del dataset 2 (FIFA - 1930.csv … FIFA - 2022.csv)
DIR = ROOT / "data" / "raw" / "dataset_2"
# Nomi del dataset 2 → nomi del dataset 1, così le due fonti si collegano per nome
ALIAS = {"Bulgaria**": "Bulgaria", "Israel*": "Israel", "USA": "United States",
         "China PR": "China", "FR Yugoslavia": "Yugoslavia"}

# Si legge ogni file e si aggiunge la colonna year, presa dal nome del file
frames = []
for f in sorted(DIR.glob("FIFA - [0-9]*.csv")):
    df = pd.read_csv(f)
    df.insert(0, "year", int(re.search(r"\d{4}", f.name).group()))
    frames.append(df)
# Un'unica tabella con tutti gli anni (489 righe)
st = pd.concat(frames, ignore_index=True)

# Il segno meno tipografico (U+2212) impedisce la conversione in numero: si sostituisce
st["Goal Difference"] = pd.to_numeric(st["Goal Difference"].astype(str).str.replace("\u2212", "-"))
st["team_name"] = st["Team"].replace(ALIAS)
# Punti per vittoria dell'epoca: 2 fino al 1990, 3 dal 1994
st["points_system"] = st["year"].map(lambda y: 2 if y < 1994 else 3)

# Controlli di coerenza: differenza reti e partite giocate
assert (st["Goals For"] - st["Goals Against"] == st["Goal Difference"]).all()
assert (st.Win + st.Draw + st.Loss == st["Games Played"]).all()

## Dataset 3
Statistiche dei giocatori del 2022: valori mancanti, tipi, quattro formati di data, emoji nei nomi, refusi delle marche.

In [ ]:
# [Cella 3] Dataset 3: lettura del file
import numpy as np
import pandas as pd

DIR3 = ROOT / "data" / "raw" / "dataset_3"
# Statistiche dei giocatori del Mondiale 2022 (814 righe, una per giocatore)
pl = pd.read_csv(DIR3 / "FIFA WC 2022 Players Stats.csv", encoding="utf-8")

In [ ]:
# [Cella 4] Dataset 3: pulizia di valori, tipi, date, nomi e marche
import re
# Spazi in più nei nomi delle colonne e nei valori di testo
pl.columns = pl.columns.str.strip()
testo = pl.select_dtypes(include=["object", "string"]).columns
pl[testo] = pl[testo].apply(lambda s: s.str.strip())

# Nelle presenze il trattino vuol dire 0; negli altri campi è un valore mancante
pl["Appearances"] = pl["Appearances"].replace("-", "0")
pl = pl.replace({"-": np.nan, "N.A": np.nan, "N,A": np.nan})

# Numeri: interi Int64 (ammettono i mancanti) per i conteggi, decimali per le medie su 90'
conteggi = ["Appearances", "Goals Scored", "Assists Provided"]
per90 = ["Dribbles per 90", "Interceptions per 90", "Tackles per 90", "Total Duels Won per 90"]
pl[conteggi] = pl[conteggi].apply(pd.to_numeric).astype("Int64")
pl[per90] = pl[per90].apply(pd.to_numeric)

# Percentuali '71.4%' → frazioni tra 0 e 1
for c in ["Save Percentage", "Clean Sheets"]:
    pl[c] = pd.to_numeric(pl[c].str.rstrip("%")) / 100
pl = pl.rename(columns={"Clean Sheets": "Clean Sheet Rate"})

# Date in quattro formati ('Jun 24, 1987', 'Sept …'): format="mixed" le legge tutte
pl["Player DOB"] = pd.to_datetime(pl["Player DOB"].str.replace("Sept ", "Sep ", regex=False), format="mixed")
pl["National Team Jersey Number"] = pl["National Team Jersey Number"].astype("Int64")
# Toglie emoji e simboli dal nome con re di Python, che conserva le lettere accentate
pl["Player Name"] = pl["Player Name"].map(lambda s: re.sub(r"[^\w\s'.\-]", "", s).strip())

# Refusi delle marche corretti a mano
BRAND = {"Nke": "Nike", "NIke": "Nike", "Pirma": "Puma", "Pima": "Puma"}
for c in ["National Team Kit Sponsor", "Brand Sponsor/Brand Used"]:
    pl[c] = pl[c].replace(BRAND)
# Nome della nazionale come nel dataset 1: serve al blocking dell'entity resolution
pl["team_name"] = pl["Nationality"].replace({"USA": "United States"})

## Immagini (dataset 4)
Indice di tutti i file con formato reale e impronta md5; nomi delle cartelle riparati (UTF-8 letto come cp850 durante l'estrazione).

In [ ]:
# [Cella 5] Immagini: cartella del dataset
# hashlib calcola l'impronta md5 di ogni file: file identici hanno la stessa impronta
import hashlib
import pandas as pd

# Struttura: Group X / <Squadra> Players / Images_<giocatore> / file
DIR4 = ROOT / "data" / "raw" / "dataset_4" / "Images" / "Images"

In [ ]:
# [Cella 6] Immagini: riparazione dei nomi e formato reale del file
# Nomi delle cartelle estratti male (UTF-8 letto come cp850, 'Mbapp├®'): si inverte l'errore
def ripara(nome):
    try:
        return nome.encode("cp850").decode("utf-8")
    # Se il nome era già corretto la conversione fallisce: si lascia com'è
    except (UnicodeEncodeError, UnicodeDecodeError):
        return nome


# Formato vero dai primi byte del file (magic number), non dall'estensione
def formato_reale(contenuto):
    if contenuto.startswith(b"\xff\xd8\xff"):
        return "JPEG"
    if contenuto.startswith(b"\x89PNG"):
        return "PNG"
    return "ALTRO"

In [ ]:
# [Cella 7] Immagini: indice di tutti i file con impronta md5
# Visita girone → squadra → giocatore → file: una riga per immagine
righe = []
for cartella_gruppo in sorted(DIR4.iterdir()):
    for cartella_squadra in sorted(cartella_gruppo.iterdir()):
        for cartella_giocatore in sorted(cartella_squadra.iterdir()):
            for img in cartella_giocatore.iterdir():
                # Il file si legge una volta: servono dimensione, formato e md5
                contenuto = img.read_bytes()
                righe.append({
                    "group": cartella_gruppo.name,
                    "team_folder": cartella_squadra.name,
                    "player_folder": cartella_giocatore.name,
                    "file": img.name,
                    "path": str(img.relative_to(DIR4)),
                    "bytes": len(contenuto),
                    "format": formato_reale(contenuto),
                    "md5": hashlib.md5(contenuto).hexdigest(),
                })
imgs = pd.DataFrame(righe)

In [ ]:
# [Cella 8] Immagini: giocatore, capitano, squadra e girone dal nome della cartella
# Nome del giocatore dalla cartella: riparato e senza il prefisso Images_
imgs["player_name"] = (imgs["player_folder"].map(ripara)
                       .str.replace(r"^Images_", "", regex=True))
# Il capitano è scritto solo nel nome della cartella: '(captain)'
imgs["captain"] = imgs["player_name"].str.contains("(captain)", regex=False)
imgs["player_name"] = imgs["player_name"].str.replace(r"\s*\(captain\)", "", regex=True).str.strip()
# 'Argentina Players' → 'Argentina'; un alias per l'Olanda
imgs["team_name"] = (imgs["team_folder"].str.replace(r"\s+players$", "", case=False, regex=True)
                     .replace({"Netherland": "Netherlands"}))
imgs["group"] = imgs["group"].str.replace("Group ", "", regex=False)

In [ ]:
# [Cella 9] Immagini: cartella dei dati elaborati e primo salvataggio dell'indice
# Cartella dei dati elaborati, letta da tutti i notebook successivi
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
# Primo salvataggio dell'indice; la cella 17 lo riscrive con i player_id
imgs.to_csv(OUT / "images_index.csv", index=False, encoding="utf-8")

## Entity resolution (Op. 11)
I giocatori del dataset 3 e delle cartelle delle immagini si collegano ai `player_id` del dataset 1 con un abbinamento a cascata, sempre dentro la stessa nazionale (*blocking*): nome esatto → nome simile (`token_sort_ratio` ≥ 85) → data di nascita e maglia → correzione manuale documentata.

In [ ]:
# [Cella 10] Entity resolution: convocati 2022 del dataset 1
import re
import pandas as pd
from unidecode import unidecode
from rapidfuzz import fuzz

# I convocati 2022 del dataset 1: i candidati a cui abbinare i nomi
rose = pd.read_sql("""
    SELECT s.player_id, t.team_name, p.given_name, p.family_name,
           p.birth_date, s.shirt_number
    FROM squads s
    JOIN teams t   USING (team_id)
    JOIN players p USING (player_id)
    WHERE s.tournament_id = 'WC-2022'""", con)
# Data come datetime, per confrontarla con quella del dataset 3
rose["birth_date"] = pd.to_datetime(rose["birth_date"], errors="coerce")

In [ ]:
# [Cella 11] Entity resolution: normalizzazione dei nomi
# Nome confrontabile: senza accenti, minuscolo, solo lettere ('Luka Modrić' → 'luka modric')
def normalizza(nome):
    nome = unidecode(str(nome)).lower().replace("not applicable", "")
    nome = re.sub(r"[^a-z ]", " ", nome)
    return " ".join(nome.split())

# Nome completo normalizzato di ogni convocato
rose["nome_norm"] = (rose["given_name"] + " " + rose["family_name"]).map(normalizza)

In [ ]:
# [Cella 12] Entity resolution: abbinamento a cascata con blocking per nazionale
# Abbinamento a cascata: per ogni riga si prova il criterio più sicuro, poi il successivo
def abbina(df, col_nome, col_dob=None, col_maglia=None, soglia=85):
    risultati = []
    for idx, riga in df.iterrows():
        # Blocking: si confronta solo con i convocati della stessa nazionale
        candidati = rose[rose["team_name"] == riga["team_name"]]
        nome = normalizza(riga[col_nome])

        # 1) nome normalizzato identico
        esatti = candidati[candidati["nome_norm"] == nome]
        if len(esatti) == 1:
            risultati.append((idx, esatti["player_id"].iat[0], "1_nome_esatto", 100))
            continue

        # 2) nome simile: token_sort_ratio ignora l'ordine delle parole; soglia 85 su 100
        punteggi = candidati["nome_norm"].map(lambda c: fuzz.token_sort_ratio(nome, c))
        if len(punteggi) and punteggi.max() >= soglia:
            migliore = punteggi.idxmax()
            risultati.append((idx, candidati.at[migliore, "player_id"], "2_fuzzy", round(punteggi.max())))
            continue

        # 3) stessa data di nascita e stessa maglia (solo per il dataset 3)
        if col_dob and col_maglia:
            stessi = candidati[(candidati["birth_date"] == riga[col_dob]) &
                               (candidati["shirt_number"] == riga[col_maglia])]
            if len(stessi) == 1:
                risultati.append((idx, stessi["player_id"].iat[0], "3_nascita_e_maglia", None))
                continue

        # 4) nessun criterio soddisfatto: da verificare a mano
        risultati.append((idx, None, "4_da_verificare", round(punteggi.max()) if len(punteggi) else None))

    # Per ogni riga in ingresso: player_id, metodo usato, punteggio
    return pd.DataFrame(risultati, columns=["idx", "player_id", "metodo", "score"]).set_index("idx")

In [ ]:
# [Cella 13] Abbinamento dei giocatori del dataset 3
# Dataset 3: 814 righe, con data di nascita e maglia come terzo criterio
m_pl = abbina(pl, "Player Name", col_dob="Player DOB", col_maglia="National Team Jersey Number")
# Si aggiungono player_id, metodo e punteggio (drop: la cella si può rieseguire)
pl = pl.drop(columns=["player_id", "metodo", "score"], errors="ignore").join(m_pl)
print(pl["metodo"].value_counts().sort_index())

metodo
1_nome_esatto         766
2_fuzzy                28
3_nascita_e_maglia     19
4_da_verificare         1
Name: count, dtype: int64


In [ ]:
# [Cella 14] Dataset 3: correzione manuale documentata
# Correzione documentata: 'Papu' Gómez ha nel dataset 3 una data di nascita sbagliata
CORREZIONI_PL = {"Alejandro Gomez": "P-10231"}

sel = pl["Player Name"].isin(CORREZIONI_PL.keys())
pl.loc[sel, "player_id"] = pl.loc[sel, "Player Name"].map(CORREZIONI_PL)
pl.loc[sel, "metodo"] = "5_manuale"

print(pl.loc[sel, ["Player Name", "team_name", "player_id", "metodo"]])

        Player Name  team_name player_id     metodo
18  Alejandro Gomez  Argentina   P-10231  5_manuale


In [ ]:
# [Cella 15] Abbinamento delle cartelle delle immagini
# Una riga per cartella (831), non per immagine
cartelle = (imgs.drop_duplicates(["team_name", "player_folder"])
                [["team_name", "player_folder", "player_name", "captain"]]
                .reset_index(drop=True))
# Stessa cascata, senza data e maglia: delle cartelle si conosce solo il nome
cartelle = cartelle.join(abbina(cartelle, "player_name"))

# Sadio Mané: nella rosa pre-torneo, non in quella finale del dataset 1
CORREZIONI_IMG = {"Sadio Mané": "P-79299"}
sel = cartelle["player_name"].isin(CORREZIONI_IMG.keys())
cartelle.loc[sel, "player_id"] = cartelle.loc[sel, "player_name"].map(CORREZIONI_IMG)
cartelle.loc[sel, "metodo"] = "5_manuale"

print(len(cartelle))
print(cartelle["metodo"].value_counts().sort_index())

831
metodo
1_nome_esatto      821
2_fuzzy              8
4_da_verificare      1
5_manuale            1
Name: count, dtype: int64


In [ ]:
# [Cella 16] player_id su tutte le immagini
# Il player_id della cartella passa a tutte le sue immagini
imgs = imgs.drop(columns=["player_id", "metodo"], errors="ignore").merge(
    cartelle[["team_name", "player_folder", "player_id", "metodo"]],
    on=["team_name", "player_folder"], how="left")

print(imgs.shape)
print("immagini senza player_id:", imgs["player_id"].isna().sum())

(41510, 13)
immagini senza player_id: 50


## Salvataggio
Abbinamenti e indice delle immagini in CSV; i dataset 2 e 3 in Parquet, che conserva i tipi delle colonne (date, interi con valori mancanti).

In [ ]:
# [Cella 17] Salvataggio degli abbinamenti e dell'indice delle immagini
# Abbinamenti salvati: li usano i notebook successivi
pl[["Player Name", "team_name", "player_id", "metodo", "score"]].to_csv(
    OUT / "match_dataset_3.csv", index=False, encoding="utf-8")
cartelle.to_csv(OUT / "match_images.csv", index=False, encoding="utf-8")
imgs.to_csv(OUT / "images_index.csv", index=False, encoding="utf-8")

print(sorted(p.name for p in OUT.iterdir()))

['images_index.csv', 'match_dataset_3.csv', 'match_images.csv']


In [ ]:
# [Cella 18] Salvataggio dei dataset 2 e 3 in Parquet
# Parquet conserva i tipi (date, interi con mancanti): il notebook 02 riparte da qui
st.to_parquet(OUT / "dataset_2_clean.parquet", index=False)
pl.to_parquet(OUT / "dataset_3_clean.parquet", index=False)

print(pd.read_parquet(OUT / "dataset_2_clean.parquet").shape,
      pd.read_parquet(OUT / "dataset_3_clean.parquet").dtypes["Player DOB"])

(489, 13) datetime64[us]
